# Notebook 08: Hybrid Allergen Detection

## Overview

Combines rule-based and ML predictions for robust allergen detection.

## Workflow

1. Load pre-trained model using utility modules
2. Load hybrid configuration
3. Define rule-based patterns (import from text_processing)
4. Hybrid prediction logic
5. Evaluate on test set using utility evaluation functions
6. Save configuration

## 0. Setup & Imports

Using modular utility functions to reduce code duplication

In [1]:
import json
import torch
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, f1_score

# Import project utility modules
from utils.data_utils import (
    load_labeled_data,
    create_stratified_splits,
    get_data_directories
)
from utils.text_processing import (
    rule_match,
    get_allergen_list,
    clean_html,
    allergens_to_binary
)
from utils.model_utils import (
    load_model_and_tokenizer,
    load_hybrid_config,
    predict_ml,
    hybrid_predict,
    find_best_thresholds
)
from utils.evaluation import (
    print_classification_report,
    print_per_class_metrics,
    error_analysis
)

# Set seeds for reproducibility
import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Imports completed successfully using utility modules!")

Imports completed successfully using utility modules!


## Setup Complete

All imports consolidated in the cell above, reducing code duplication across the notebook.

In [2]:
import os

# Use project directory structure for consistent paths
dirs = get_data_directories()
DATA_PATH = os.path.join(dirs["final"], "labeled_dataset_enhanced.csv")
BEST_THRESHOLDS_PATH = os.path.join(dirs["models"], "best_thresholds.npy")
HYBRID_CONFIG_PATH = os.path.join(dirs["models"], "hybrid_config.json")
MODEL_PATH = os.path.join(dirs["models"], "mobilebert_allergen_final/")

# Batch size for inference — keep small to fit within GPU memory
BATCH_SIZE = 8

# Load labeled data using utility function
df = load_labeled_data(DATA_PATH)

# Create clean_text from ingredients_text_en (same as notebook 07)
df["ingredients_cleaned"] = df["ingredients_text_en"].apply(clean_html)

# Parse labels using utility function
df["labels"] = df["detected_allergens"].apply(allergens_to_binary)

print(f"Dataset shape: {df.shape}")
print(f"Label distribution (per class): {np.array(df['labels'].tolist()).sum(axis=0)}")

# Free any cached GPU memory before starting inference
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GiB total, "
          f"{torch.cuda.memory_allocated(0) / 1e9:.2f} GiB allocated")

Dataset shape: (1057, 13)
Label distribution (per class): [497  98  88  16 278 355  77  32]
GPU memory: 8.21 GiB total, 0.00 GiB allocated


In [3]:
# Generate stratified train/val/test splits (same approach as notebook 04)
# using the utility function from utils.data_utils

X = df["ingredients_cleaned"].values
y = np.array(df["labels"].tolist())

train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = create_stratified_splits(
    X, y,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    random_state=SEED
)

print(f"Train size: {len(train_texts)} (positive samples: {(np.array(train_labels).sum(axis=1)>0).sum()})")
print(f"Val size:   {len(val_texts)}   (positive samples: {(np.array(val_labels).sum(axis=1)>0).sum()})")
print(f"Test size:  {len(test_texts)}  (positive samples: {(np.array(test_labels).sum(axis=1)>0).sum()})")

Train size: 739 (positive samples: 483)
Val size:   159   (positive samples: 119)
Test size:  159  (positive samples: 110)


## 3. Rule‑Based System

Using the robust rule_match function from text_processing utility

In [4]:
# The rule_match function is imported from utils.text_processing
# It includes comprehensive allergen keywords and negation handling
# No need to redefine it here - using the utility version ensures consistency

# Example usage:
# rule_present = rule_match(text, "milk")
print("Using rule_match from utils.text_processing")
allergen_list = get_allergen_list()
print(f"Available allergens: {allergen_list}")

Using rule_match from utils.text_processing
Available allergens: ['milk', 'eggs', 'peanuts', 'tree_nuts', 'soy', 'wheat', 'fish', 'shellfish']


## 4. Load the Trained MobileBERT Model and Tokenizer

Using utility functions for model loading

In [5]:
# Load the trained MobileBERT model and tokenizer using utilities
# Note: The model must be trained first by running notebook 07

print(f"Loading model from {MODEL_PATH}...")
model, tokenizer, device = load_model_and_tokenizer(MODEL_PATH)

# Compute max_length from the training data lengths for consistency with notebook 07
token_lengths = [len(tokenizer.encode(t, truncation=True)) for t in df["ingredients_cleaned"]]
MAX_LENGTH = max(token_lengths)
print(f"Computed max sequence length: {MAX_LENGTH}")

# Load hybrid configuration
try:
    hybrid_config = load_hybrid_config(HYBRID_CONFIG_PATH)
    ml_thresholds = hybrid_config.get("ml_thresholds", None)
    rule_conf_threshold = hybrid_config.get("rule_conf_threshold", 0.5)
    mode = hybrid_config.get("mode", "hard_override")
    print(f"Loaded ML thresholds: {ml_thresholds}")
    print(f"Rule conf threshold: {rule_conf_threshold}")
    print(f"Mode: {mode}")
except FileNotFoundError:
    print(f"Hybrid config not found at {HYBRID_CONFIG_PATH}")
    print("Using default parameters — will compute thresholds from validation set")
    ml_thresholds = None
    rule_conf_threshold = 0.5
    mode = "hard_override"

Loading model from /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/mobilebert_allergen_final/...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Computed max sequence length: 540
Loaded ML thresholds: [0.05, 0.02, 0.35000000000000003, 0.5, 0.73, 0.13, 0.08, 0.5]
Rule conf threshold: 0.5
Mode: hard_override


## 5. Text Preprocessing Function

Simple preprocessing (can be enhanced with utilities if needed)

In [6]:
# Text preprocessing is already done in cell 6 (clean_html) and cell 11 (tokenizer)
# The model handles tokenization internally via predict_ml and hybrid_predict
print(f"Using {BATCH_SIZE} samples per batch for inference")
print(f"Using max sequence length: {MAX_LENGTH}")

Using 8 samples per batch for inference
Using max sequence length: 540


## 6. Load Optimal ML Thresholds

Using utility function to find optimal thresholds if not already saved

In [7]:
# Try to load pre-computed optimal thresholds
try:
    best_thresholds = np.load(BEST_THRESHOLDS_PATH)
    print(f"Loaded pre-computed per-class thresholds: {best_thresholds}")
except FileNotFoundError:
    print(f"Pre-computed thresholds not found at {BEST_THRESHOLDS_PATH}")
    print("Computing optimal thresholds from validation set...")
    val_ml_preds, val_probs = predict_ml(
        val_texts, model, tokenizer, device,
        max_length=MAX_LENGTH, batch_size=BATCH_SIZE
    )
    best_thresholds = find_best_thresholds(val_probs, np.array(val_labels))
    print("Computed thresholds:", best_thresholds)

Loaded pre-computed per-class thresholds: [0.24 0.95 0.84 0.01 0.92 0.82 0.96 0.4 ]


## 7. Evaluate on Test Set

Using utility evaluation functions for consistent reporting

In [8]:
# ML only baseline — uses batched inference to avoid GPU OOM
print(f"Running ML inference with batch_size={BATCH_SIZE}, max_length={MAX_LENGTH}...")
ml_test_preds, ml_test_probs = predict_ml(
    test_texts, model, tokenizer, device,
    thresholds=best_thresholds,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE
)
print("ML Only evaluation:")
print_classification_report(test_labels, ml_test_preds, get_allergen_list(), prefix="ML Only")

# Hybrid (hard override with global optimal threshold)
print("\nRunning hybrid (hard override) inference...")
test_hybrid_preds, _ = hybrid_predict(
    test_texts, model, tokenizer, device,
    thresholds=best_thresholds,
    rule_conf_threshold=rule_conf_threshold,
    mode='hard_override',
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE
)
print("Hybrid (hard override) evaluation:")
print_classification_report(test_labels, test_hybrid_preds, get_allergen_list(), prefix="Hybrid (hard override)")

# Try alternative mode: high confidence bypass
print("\nRunning hybrid (high confidence bypass) inference...")
test_hybrid_bypass, _ = hybrid_predict(
    test_texts, model, tokenizer, device,
    thresholds=best_thresholds,
    rule_conf_threshold=rule_conf_threshold,
    mode='high_confidence_bypass',
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE
)
print("Hybrid (high confidence bypass) evaluation:")
print_classification_report(test_labels, test_hybrid_bypass, get_allergen_list(), prefix="Hybrid (high confidence bypass)")

Running ML inference with batch_size=8, max_length=540...
ML Only evaluation:
=== ML Only ===
              precision    recall  f1-score   support

        milk     0.9722    0.9333    0.9524        75
        eggs     1.0000    1.0000    1.0000        14
     peanuts     1.0000    1.0000    1.0000        13
   tree_nuts     0.2000    0.6667    0.3077         3
         soy     0.9737    0.8810    0.9250        42
       wheat     1.0000    0.9444    0.9714        54
        fish     1.0000    0.5833    0.7368        12
   shellfish     0.8000    0.8000    0.8000         5

   micro avg     0.9429    0.9083    0.9252       218
   macro avg     0.8682    0.8511    0.8367       218
weighted avg     0.9698    0.9083    0.9335       218
 samples avg     0.6415    0.6266    0.6279       218

Macro F1: 0.8367
Micro F1: 0.9252


Running hybrid (hard override) inference...
Hybrid (hard override) evaluation:
=== Hybrid (hard override) ===
              precision    recall  f1-score   support



## 8. Error Analysis (examples where hybrid differs from ML)

Using utility function for error analysis

In [9]:
results_df = pd.DataFrame({
    "text": test_texts,
    "true": [list(l) for l in test_labels],
    "ml_pred": [list(p) for p in ml_test_preds],
    "hybrid_pred": [list(p) for p in test_hybrid_preds]
})
differences = results_df[results_df["ml_pred"] != results_df["hybrid_pred"]]
print(f"Number of test samples where hybrid changed prediction: {len(differences)}")
if len(differences) > 0:
    print("\nFirst 3 examples:")
    allergen_list = get_allergen_list()
    for i in range(min(3, len(differences))):
        row = differences.iloc[i]
        true_all = [allergen_list[j] for j, v in enumerate(row['true']) if v == 1]
        ml_all = [allergen_list[j] for j, v in enumerate(row['ml_pred']) if v == 1]
        hybrid_all = [allergen_list[j] for j, v in enumerate(row['hybrid_pred']) if v == 1]
        print(f"\nText: {row['text'][:150]}...")
        print(f"True: {true_all}")
        print(f"ML:   {ml_all}")
        print(f"Hybrid:{hybrid_all}")

Number of test samples where hybrid changed prediction: 8

First 3 examples:

Text: dark chocolate coating: sugar, hydrogenated vegetable oil, cocoa powder, emulsifiers (soya lecithin (e322), polyglycerol polyricinoleate (e476)], natu...
True: ['milk', 'peanuts', 'soy', 'wheat']
ML:   ['milk', 'peanuts', 'soy', 'wheat']
Hybrid:['milk', 'eggs', 'peanuts', 'soy', 'wheat']

Text: wheat flour, vegetable oil, acetylated distarch adipate, iodized salt, sodium polyphosphate, sodium carboxymethyl cellulose, acidity regulators, artif...
True: ['wheat', 'fish']
ML:   ['wheat']
Hybrid:['wheat', 'fish']

Text: water, skimmed milk, lactose, vegetable oils (palm olein, coconut, canola, sunflower), galacto-oligosaccharides (gos) (from milk), whey protein concen...
True: ['milk', 'fish']
ML:   ['milk']
Hybrid:['milk', 'fish']


## 9. Save hybrid configuration for production

Saving the current hybrid configuration

In [10]:
hybrid_config = {
    "ml_thresholds": best_thresholds.tolist(),
    "rule_conf_threshold": rule_conf_threshold,
    "mode": mode
}
config_path = os.path.join(dirs["models"], "hybrid_config.json")
with open(config_path, "w") as f:
    json.dump(hybrid_config, f, indent=2)
print(f"Hybrid configuration saved to {config_path}")

Hybrid configuration saved to /home/Woof/Dev/ENHANCING-FOOD-LABEL-TRANSPARENCY-FOR-FILIPINO-CONSUMERS-THROUGH-AI-BASED-INGREDIENT-INTERPRETATION/models/hybrid_config.json
